# Bronze Layer

### All Tables

In [3]:
tables = {
    "appointments": "appointment_id",
    "patients": "patient_id",
    "doctors": "doctor_id",
    "clinics": "clinic_id",
    "treatments": "treatment_id",
    "payments": "payment_id",
    "prescriptions":"prescription_id"
}

StatementMeta(, 53258627-0a4d-4730-bc43-91fa7803ccac, 5, Finished, Available, Finished, False)

### Import Source Function

In [1]:
from pyspark.sql.functions import current_timestamp, input_file_name

def read_source(table_name):
    return (
        spark.read
             .option("header", True)
             .option("inferSchema", True)
             .csv(f"Files/{table_name}.csv")
             .withColumn("load_datetime", current_timestamp())
             .withColumn("source_file", input_file_name())
    )

StatementMeta(, 53258627-0a4d-4730-bc43-91fa7803ccac, 3, Finished, Available, Finished, False)

In [13]:
## Source for text file.
def read_source_txt():
    return (
        spark.read
         .option("header", False)
         .option("inferSchema", True)
         .option("delimiter", "|")
         .csv("Files/patient_insurance.txt")
         .toDF(
             "patient_id",
             "insurance_status",
             "insurance_provider",
             "coverage_level"
         )
         .withColumn("load_datetime", current_timestamp())
         .withColumn("source_file", input_file_name())
    )

StatementMeta(, 53258627-0a4d-4730-bc43-91fa7803ccac, 15, Finished, Available, Finished, False)

### Create Tables in Bronze Layer Function

In [4]:
# def create_bronze_table(table_name):

#     df = read_source(table_name)

#     if not spark.catalog.tableExists(f"bronze.{table_name}"):

#         (
#             df.limit(0)
#               .write
#               .format("delta")
#               .saveAsTable(f"bronze.{table_name}")
#         )

#         print(f"{table_name} table created successfully.")

#     else:
#         print(f"{table_name} table already exists.")

StatementMeta(, 53258627-0a4d-4730-bc43-91fa7803ccac, 6, Finished, Available, Finished, False)

### Text File to Table

In [14]:
# insurance = read_source_txt()

# if not spark.catalog.tableExists("bronze.insurance"):
#     (
#         insurance.limit(0)
#                  .write
#                  .format("delta")
#                  .saveAsTable("bronze.insurance")
#     )

#     print("Insurance Table Created Successfully.")

# else:
#     print("Insurance Table Already Exists.")

StatementMeta(, 53258627-0a4d-4730-bc43-91fa7803ccac, 16, Finished, Available, Finished, False)

Insurance Table Already Exists.


### Calling Bronze function

In [6]:
# for table in tables:
#     create_bronze_table(table)

StatementMeta(, 53258627-0a4d-4730-bc43-91fa7803ccac, 8, Finished, Available, Finished, False)

appointments table created successfully.
patients table created successfully.
doctors table created successfully.
clinics table created successfully.
treatments table created successfully.
payments table created successfully.
prescriptions table created successfully.


### Incremental Load To Bronze

In [9]:
from delta.tables import DeltaTable
def incremental_load(table_name, primary_key):

    src_df = read_source(table_name)

    tgt = DeltaTable.forName(spark, f"bronze.{table_name}")

    (
        tgt.alias("tgt")
           .merge(
               src_df.alias("src"),
               f"src.{primary_key} = tgt.{primary_key}"
           )
           .whenNotMatchedInsertAll()
           .execute()
    )

    print(f"{table_name} Incremental Load Completed.")

StatementMeta(, 53258627-0a4d-4730-bc43-91fa7803ccac, 11, Finished, Available, Finished, False)

In [10]:
for table_name, primary_key in tables.items():
    incremental_load(table_name, primary_key)

StatementMeta(, 53258627-0a4d-4730-bc43-91fa7803ccac, 12, Finished, Available, Finished, False)

appointments Incremental Load Completed.
patients Incremental Load Completed.
doctors Incremental Load Completed.
clinics Incremental Load Completed.
treatments Incremental Load Completed.
payments Incremental Load Completed.
prescriptions Incremental Load Completed.


### Incremental For Text File In Bronze

In [15]:
from delta.tables import DeltaTable

src_df = read_source_txt()

tgt = DeltaTable.forName(spark, f"bronze.insurance")

(
    tgt.alias("tgt")
        .merge(
            src_df.alias("src"),
            f"src.patient_id = tgt.patient_id"
        )
        .whenNotMatchedInsertAll()
        .execute()
)

print(f"Insurance Incremental Load Completed.")

StatementMeta(, 53258627-0a4d-4730-bc43-91fa7803ccac, 17, Finished, Available, Finished, False)

Insurance Incremental Load Completed.
